In [1]:
from langchain_community.vectorstores import FAISS
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import TextLoader

C:\Users\mrraj\AppData\Local\Temp\ipykernel_9724\1723707667.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
loader = TextLoader("rag_knowledge.txt")
docs = loader.load()
docs

[Document(metadata={'source': 'rag_knowledge.txt'}, page_content='Retrieval Augmented Generation is commonly known as RAG.\nRAG combines information retrieval with large language models.\nA RAG system retrieves relevant information before generating an answer.\nThe retrieved information is provided to the language model as context.\nRAG is useful when a language model needs access to external knowledge.\nExternal knowledge can include documents, websites, databases, manuals, reports, and company information.\nRAG can reduce the need to retrain a language model whenever new information becomes available.\nA typical RAG pipeline contains document loading, text splitting, embedding generation, vector storage, retrieval, and generation.\nSome advanced RAG systems also include query enhancement, reranking, hybrid search, filtering, and evaluation.\nThe quality of retrieval has a significant impact on the quality of the final answer.\nIf the retriever returns irrelevant documents, the langua

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)
chunks = splitter.split_documents(docs)
chunks

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name = "all-MiniLM-L6-v2"
)
vectorestore = FAISS.from_documents(chunks,embeddings)
retriever = vectorestore.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":5}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llm = init_chat_model(
    model="groq:openai/gpt-oss-120b"
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000293879AC590>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000293879AE900>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

decomposition_prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful query decomposition assistant.

    Break the user's complex question into smaller and simpler
    questions that can be answered independently.

    Original Question:
    {query}

    Return only the sub-questions, one per line.
    """
)
decomposition_chain = decomposition_prompt | llm | StrOutputParser()


In [8]:
query = """
What is RAG, how does it work, and what is the difference
between RAG and fine-tuning?
"""

decomposition_response = decomposition_chain.invoke({
    "query": query
})

print(decomposition_response)

What is Retrieval‑Augmented Generation (RAG)?  
How does Retrieval‑Augmented Generation (RAG) work?  
What is the difference between RAG and fine‑tuning?


In [13]:
from langchain_core.prompts import ChatPromptTemplate

main_prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful AI assistant.

    Answer the user's question using the provided context.

    Context:
    {context}

    Question:
    {input}

    Instructions:
    - Answer only from the given context.
    - If the answer is not present in the context, say you don't know.
    - Keep the answer clear and concise.

    Answer:
    """
)

qa_chain = create_stuff_documents_chain(llm=llm,prompt=main_prompt)

In [ ]:
sub_questions = decomposition_response.split("\n")

sub_questions = [
    q.strip()
    for q in sub_questions
    if q.strip()
]

print("Sub Questions:")
for i, q in enumerate(sub_questions, 1):
    print(f"{i}. {q}")

Sub Questions:
1. What is Retrieval‑Augmented Generation (RAG)?
2. How does Retrieval‑Augmented Generation (RAG) work?
3. What is the difference between RAG and fine‑tuning?


In [10]:
all_docs = []

for question in sub_questions:
    docs = retriever.invoke(question)
    all_docs.extend(docs)

In [14]:
original_query = """
What is RAG, how does it work, and what is the difference
between RAG and fine-tuning?
"""

response = qa_chain.invoke({
    "input": original_query,
    "context": all_docs
})

print("\n" + "=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(response)


FINAL ANSWER
**RAG (Retrieval‑Augmented Generation)** is a technique that combines information retrieval with a large language model.  
1. **Document collection** – Useful documents are first gathered from sources such as PDFs, text files, web pages, databases, or cloud storage using a document loader.  
2. **Retrieval** – When a user asks a question, a retriever selects relevant documents (often via vector similarity search). Techniques like Maximal Marginal Relevance (MMR) can be used to balance relevance and diversity of the retrieved chunks.  
3. **Generation** – The retrieved documents are supplied to the language model as context, allowing it to generate an answer that is grounded in the external evidence. This helps reduce hallucination, though it does not eliminate it entirely.

**Difference between RAG and fine‑tuning**  
The provided context does not contain a description of fine‑tuning or a direct comparison between RAG and fine‑tuning. Therefore, I don’t know the differenc